# Dave's Trading System: Bounce 5, Predator Play, Fibonacci & Trinity

**Instruments:** SPY (ES equivalent), QQQ (NQ equivalent) — Alpaca paper trading

**Timeframe:** 1-min or 5-min bars

**Moving Averages:** 50-period MA (short-term) | 200-period MA (long-term)

---

## Strategies
| # | Strategy | Season | Win Rate |
|---|---|---|---|
| 1 | Season Detector | All | Foundation |
| 2 | Bounce 5 Channel Play | Season 1 Ranging | 78% |
| 3 | Predator Play | Season 2 Trending | 90% |
| 4 | Fibonacci Retracement | All | High |
| 5 | Trinity Play | Convergence | A+ setup |

> Paper trading only. Always test before going live.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JaBoyWyatt/alpaca-py/blob/master/examples/stocks/bounce5-trading-system.ipynb)

## 1. Install & Import

In [ ]:
!pip install alpaca-py pandas numpy matplotlib scipy --quiet

In [ ]:
import os, math, time, datetime, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.stats import linregress
from alpaca.trading.client import TradingClient
from alpaca.trading.requests import MarketOrderRequest, LimitOrderRequest, GetOrdersRequest
from alpaca.trading.enums import OrderSide, TimeInForce, OrderStatus
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame, TimeFrameUnit
print('Imports OK')

## 2. Configuration

In [ ]:
# ── API Credentials (paper trading) ──────────────────────────────────────────
# Get your keys at: https://app.alpaca.markets/paper/dashboard/overview
API_KEY    = os.environ.get('ALPACA_API_KEY',    'YOUR_PAPER_API_KEY')
API_SECRET = os.environ.get('ALPACA_API_SECRET', 'YOUR_PAPER_SECRET_KEY')
PAPER      = True

# ── Instrument ───────────────────────────────────────────────────────────────
SYMBOL     = 'SPY'   # SPY = ES equivalent | QQQ = NQ equivalent
TIMEFRAME  = '1Min'  # '1Min' or '5Min'

# ── Moving Averages ──────────────────────────────────────────────────────────
MA_SHORT   = 50    # Cyan on chart
MA_LONG    = 200   # White on chart

# ── Risk Management ──────────────────────────────────────────────────────────
DAILY_GOAL_USD     = 100     # Stop trading after this profit (ES = $100/contract)
RISK_PER_TRADE_PCT = 0.005   # 0.5% of account per trade
MAX_TRADES_PER_DAY = 3
MIN_RISK_REWARD    = 4.0     # Bounce 5 minimum R:R

# ── Bounce 5 Channel ─────────────────────────────────────────────────────────
MAX_CHANNEL_ANGLE  = 45      # Reject channels steeper than this
CHANNEL_LOOKBACK   = 50      # Bars to look back

# ── Fibonacci ────────────────────────────────────────────────────────────────
FIB_LEVELS = [0.382, 0.500, 0.618]

# ── Trinity confluence tolerance ─────────────────────────────────────────────
TRINITY_TOL = 0.002   # 0.2% price range for confluence

print(f'Config loaded | {SYMBOL} | {TIMEFRAME} | Paper={PAPER}')

## 3. Alpaca Client Setup

In [ ]:
trading_client = TradingClient(API_KEY, API_SECRET, paper=PAPER)
data_client    = StockHistoricalDataClient(API_KEY, API_SECRET)
account = trading_client.get_account()
print(f'Account status : {account.status}')
print(f'Buying power   : ${float(account.buying_power):,.2f}')
print(f'Portfolio value: ${float(account.portfolio_value):,.2f}')

## 4. Data Fetching & Indicators

In [ ]:
def get_bars(symbol, timeframe_str='1Min', lookback_bars=400):
    """Fetch OHLCV bars and compute 50/200 MAs plus slopes."""
    tf_map = {
        '1Min':  TimeFrame(1,  TimeFrameUnit.Minute),
        '5Min':  TimeFrame(5,  TimeFrameUnit.Minute),
        '15Min': TimeFrame(15, TimeFrameUnit.Minute),
    }
    tf    = tf_map.get(timeframe_str, TimeFrame(1, TimeFrameUnit.Minute))
    start = datetime.datetime.utcnow() - datetime.timedelta(days=5)
    req   = StockBarsRequest(symbol_or_symbols=symbol, timeframe=tf,
                             start=start, limit=lookback_bars)
    bars  = data_client.get_stock_bars(req).df
    if isinstance(bars.index, pd.MultiIndex):
        bars = bars.xs(symbol, level='symbol')
    bars  = bars[['open','high','low','close','volume']].copy()
    bars.index = pd.to_datetime(bars.index)
    bars.sort_index(inplace=True)
    bars['ma50']        = bars['close'].rolling(MA_SHORT).mean()
    bars['ma200']       = bars['close'].rolling(MA_LONG).mean()
    bars['ma50_slope']  = bars['ma50'].diff(10)  / bars['ma50']
    bars['ma200_slope'] = bars['ma200'].diff(10) / bars['ma200']
    return bars.dropna(subset=['ma50','ma200'])

bars = get_bars(SYMBOL, TIMEFRAME)
print(f'Loaded {len(bars)} bars for {SYMBOL}')
print(bars.tail(3)[['close','ma50','ma200']])

## 5. Season Detector
| Season | Name | Condition | Strategy |
|---|---|---|---|
| 1 | RANGING | 50 & 200 MA flat, intertwined | Bounce 5 |
| 2 | TRENDING | MAs clearly separated | Predator Play |
| 3 | WHIPSAW | Price violently crossing both MAs | Stand aside |

In [ ]:
def detect_season(bars, slope_thresh=0.0015, sep_thresh=0.005, whipsaw_crosses=4, lookback=20):
    """
    Returns dict: season (1/2/3), name, description, trend_dir, golden_cross, death_cross
    Season 1 = RANGING  -> Use Bounce 5
    Season 2 = TRENDING -> Use Predator Play
    Season 3 = WHIPSAW  -> Stand aside
    """
    recent = bars.tail(lookback).copy()
    latest = bars.iloc[-1]
    prev   = bars.iloc[-2]

    ma50_slope   = abs(latest['ma50_slope'])
    ma_sep       = abs(latest['ma50'] - latest['ma200']) / latest['ma200']
    above_200    = (recent['close'] > recent['ma200']).astype(int)
    crosses_200  = above_200.diff().abs().sum()

    golden_cross = (prev['ma50'] <= prev['ma200']) and (latest['ma50'] > latest['ma200'])
    death_cross  = (prev['ma50'] >= prev['ma200']) and (latest['ma50'] < latest['ma200'])

    # Season 3: Whipsaw
    if crosses_200 >= whipsaw_crosses and ma_sep < sep_thresh:
        return {'season':3,'name':'WHIPSAW',
                'description':'Price violently crossing MAs. STAND ASIDE.',
                'trend_dir':'flat','golden_cross':golden_cross,'death_cross':death_cross}
    # Season 2: Trending
    if ma_sep >= sep_thresh or ma50_slope > slope_thresh:
        name = 'TRENDING_UP' if latest['ma50'] > latest['ma200'] else 'TRENDING_DOWN'
        td   = 'up'          if latest['ma50'] > latest['ma200'] else 'down'
        return {'season':2,'name':name,
                'description':f'MAs separated {ma_sep:.2%}. Use Predator Play.',
                'trend_dir':td,'golden_cross':golden_cross,'death_cross':death_cross}
    # Season 1: Ranging
    return {'season':1,'name':'RANGING',
            'description':'MAs flat and intertwined. Use Bounce 5 Channel Play.',
            'trend_dir':'flat','golden_cross':golden_cross,'death_cross':death_cross}

season = detect_season(bars)
emoji  = {1:'RANGE',2:'TREND',3:'WHIP'}[season['season']]
print(f'[{emoji}] Season {season["season"]}: {season["name"]}')
print(f'   {season["description"]}')
if season['golden_cross']: print('   *** GOLDEN CROSS detected!')
if season['death_cross']:  print('   *** DEATH CROSS detected!')
print(f'   50 MA  = {bars.iloc[-1]["ma50"]:.2f}')
print(f'   200 MA = {bars.iloc[-1]["ma200"]:.2f}')
print(f'   Price  = {bars.iloc[-1]["close"]:.2f}')
above50  = bars.iloc[-1]['close'] > bars.iloc[-1]['ma50']
print(f'   Price vs 50 MA : {"ABOVE -> prefer longs" if above50 else "BELOW -> prefer shorts"}')

## 6. Strategy 1: Bounce 5 Channel Play (78% win rate)
**Only trade in Season 1 (Ranging)**

```
Channel:  B1(top) -> B2(bottom) -> B3(top) -> B4(bottom) -> B5(TOP) <- SHORT HERE
Target  : B4 support line
Stop    : High above the channel
R:R     : Minimum 4:1, typical 7:1
```

**Rules:**
- Channel angle must be 45 degrees or LESS
- Channel must come from prior downward move, then ascending recovery
- Do NOT short an upward tube built from an upward move (that is bullish)
- If B4 forms a double bottom with B2: reduce size or skip
- Newbie Flush: price may pierce support 1 tick before reversing — do not place stops right at the line
- Panzy stop: once in profit, move stop to breakeven immediately

In [ ]:
def find_swing_points(bars, lookback=50, prominence=3):
    """Find swing highs and lows. Returns {'highs': [(i,price),...], 'lows': [...]}"""
    df = bars.tail(lookback).reset_index(drop=True)
    highs, lows = [], []
    for i in range(prominence, len(df) - prominence):
        wh = df['high'].iloc[i-prominence:i+prominence+1]
        wl = df['low'].iloc[i-prominence:i+prominence+1]
        if df['high'].iloc[i] == wh.max(): highs.append((i, df['high'].iloc[i]))
        if df['low'].iloc[i]  == wl.min():  lows.append((i, df['low'].iloc[i]))
    return {'highs': highs, 'lows': lows}

def channel_angle_deg(x1,y1,x2,y2, bar_ratio=0.01):
    dx = x2-x1
    dy = (y2-y1)/(y1*bar_ratio)
    return abs(math.degrees(math.atan2(dy,dx)))

def detect_bounce5(bars, lookback=CHANNEL_LOOKBACK):
    """
    Detect Bounce 5 short setup.
    Returns dict with: valid, entry_price, target_price, stop_price,
                       risk_reward, channel_angle, warnings, reason
    """
    res = {'valid':False,'warnings':[],'reason':''}
    sp  = find_swing_points(bars, lookback)
    H, L = sp['highs'], sp['lows']

    if len(H) < 3 or len(L) < 2:
        res['reason'] = f'Not enough swings (highs={len(H)}, lows={len(L)})'
        return res

    b1,b3,b5 = H[-3],H[-2],H[-1]   # 3 swing highs
    b2,b4    = L[-2],L[-1]          # 2 swing lows

    # Ascending channel check: lows must be rising
    if b4[1] <= b2[1]:
        res['reason'] = 'Bounce 4 not higher than Bounce 2. Not ascending channel.'
        return res

    # INVALID: do not short upward tube built from upward move
    # Proxy: if B1 < B2 (channel built from pure upward move), skip
    if b1[0] > b2[0] and b1[1] < b2[1]:
        res['reason'] = 'Upward tube built from upward move. Bullish — do NOT short.'
        return res

    # Channel angle check
    angle = channel_angle_deg(b2[0],b2[1],b4[0],b4[1])
    res['channel_angle'] = angle
    if angle > MAX_CHANNEL_ANGLE:
        res['reason'] = f'Channel angle {angle:.1f}deg exceeds 45deg limit.'
        return res

    # Support slope and levels
    support_slope     = (b4[1]-b2[1]) / (b4[0]-b2[0])
    current_bar       = lookback - 1
    support_now       = b4[1] + support_slope*(current_bar - b4[0])
    resistance_now    = b5[1] + support_slope*(current_bar - b5[0])

    # Double bottom warning
    if abs(b4[1]-b2[1])/b2[1] < 0.003:
        res['warnings'].append('WARNING: B4 near B2 — potential double bottom. Reduce size or skip.')

    entry_price  = resistance_now
    target_price = support_now
    stop_price   = resistance_now * 1.001   # just above channel
    stop_dist    = stop_price - entry_price
    reward       = entry_price - target_price

    if stop_dist <= 0 or reward <= 0:
        res['reason'] = 'Invalid geometry.'
        return res

    rr = reward / stop_dist
    if rr < MIN_RISK_REWARD:
        res['reason'] = f'R:R {rr:.1f}:1 below minimum {MIN_RISK_REWARD}:1'
        return res

    current_price = bars['close'].iloc[-1]
    if current_price < resistance_now * 0.998:
        res['reason'] = f'Price ${current_price:.2f} not yet at resistance ${resistance_now:.2f}. Wait.'
        return res

    res.update({'valid':True,
                'entry_price':   round(entry_price,2),
                'target_price':  round(target_price,2),
                'stop_price':    round(stop_price,2),
                'risk_reward':   round(rr,2),
                'channel_height':round(resistance_now-support_now,2),
                'support_slope': support_slope})
    return res

b5 = detect_bounce5(bars)
if b5['valid']:
    print('*** BOUNCE 5 SETUP DETECTED ***')
    print(f'  Entry (SHORT) : ${b5["entry_price"]}')
    print(f'  Target        : ${b5["target_price"]}')
    print(f'  Stop          : ${b5["stop_price"]}')
    print(f'  Risk/Reward   : {b5["risk_reward"]}:1')
    print(f'  Channel angle : {b5["channel_angle"]:.1f} deg')
    for w in b5['warnings']: print(f'  {w}')
else:
    print(f'No Bounce 5 setup: {b5["reason"]}')

## 7. Strategy 2: Predator Play (90% win rate)
**Best in Season 2 (Trending)**

```
200 MA ─────────────────────────────────
              | D (first dip below MA)
         First dip low
              | 2xD
         LONG ENTRY HERE  <- Predator flush zone
```

**Logic:** Predators flush newbie traders at the first dip, then buy at the 2x flush zone.
**Target:** Back above the 200 MA.
**Stop:** If price continues below 2xD with momentum.

In [ ]:
def detect_predator_play(bars, lookback=30):
    """
    Detect Predator Play long setup.
    Returns dict: valid, entry_price, target_price, stop_price,
                  distance_d, ma200_at_cross, risk_reward, near_entry
    """
    res    = {'valid':False,'reason':''}
    recent = bars.tail(lookback).copy().reset_index(drop=True)

    # Find most recent cross below 200 MA
    cross_idx = None
    for i in range(len(recent)-1, 0, -1):
        if (recent['close'].iloc[i-1] >= recent['ma200'].iloc[i-1] and
            recent['close'].iloc[i]   <  recent['ma200'].iloc[i]):
            cross_idx = i
            break

    if cross_idx is None:
        res['reason'] = 'No recent cross below 200 MA found.'
        return res

    ma200_at_cross = recent['ma200'].iloc[cross_idx]
    first_dip_low  = recent.iloc[cross_idx:]['low'].min()
    distance_d     = ma200_at_cross - first_dip_low

    if distance_d <= 0:
        res['reason'] = 'Price has not dipped below 200 MA yet.'
        return res

    entry_price  = ma200_at_cross - (2 * distance_d)   # 2x D below MA
    target_price = ma200_at_cross                       # back above 200 MA
    stop_price   = entry_price - (0.5 * distance_d)    # momentum exit

    reward    = target_price - entry_price
    stop_dist = entry_price  - stop_price
    rr        = reward / stop_dist if stop_dist > 0 else 0

    if rr < 2.0:
        res['reason'] = f'R:R {rr:.1f}:1 too low.'
        return res

    current   = bars['close'].iloc[-1]
    if current < entry_price:
        res['reason'] = f'Price ${current:.2f} already below entry ${entry_price:.2f}. Momentum exit.'
        return res

    near_entry = abs(current - entry_price) / entry_price < 0.003
    res.update({'valid':True,
                'entry_price':    round(entry_price,2),
                'target_price':   round(target_price,2),
                'stop_price':     round(stop_price,2),
                'distance_d':     round(distance_d,2),
                'ma200_at_cross': round(ma200_at_cross,2),
                'risk_reward':    round(rr,2),
                'near_entry':     near_entry})
    return res

pred = detect_predator_play(bars)
if pred['valid']:
    print('*** PREDATOR PLAY SETUP DETECTED ***')
    print(f'  200 MA at cross : ${pred["ma200_at_cross"]}')
    print(f'  Distance D      : ${pred["distance_d"]}')
    print(f'  Entry (LONG)    : ${pred["entry_price"]} (2xD below MA)')
    print(f'  Target          : ${pred["target_price"]} (back above 200 MA)')
    print(f'  Stop            : ${pred["stop_price"]}')
    print(f'  Risk/Reward     : {pred["risk_reward"]}:1')
    if pred['near_entry']: print('  >>> Price is near entry zone NOW!')
else:
    print(f'No Predator setup: {pred["reason"]}')

## 8. Fibonacci Retracement
Draw from swing Low (A) to swing High (B). Enter LONG on pullback:
- 38.2% = first / shallow entry (lower probability)
- 50.0% = second entry (moderate probability)
- **61.8% = deepest entry, HIGHEST probability (wholesale zone)**

If price closes below 61.8% with momentum = setup invalid, exit.

In [ ]:
def calc_fib_levels(swing_low, swing_high):
    """Calculate Fibonacci retracement levels from A (low) to B (high)."""
    move   = swing_high - swing_low
    levels = {}
    for f in FIB_LEVELS:
        levels[f] = round(swing_high - f * move, 2)
    levels[0.0] = round(swing_high, 2)
    levels[1.0] = round(swing_low,  2)
    return levels

def detect_fib_entry(bars, lookback=60):
    """
    Auto-detect recent A->B swing and compute Fibonacci levels.
    Returns dict: levels, current_zone, setup_valid, swing_low, swing_high
    """
    df         = bars.tail(lookback).copy()
    swing_low  = df['low'].min()
    swing_high = df['high'].max()
    current    = bars['close'].iloc[-1]
    levels     = calc_fib_levels(swing_low, swing_high)

    # Find which Fib zone current price is in
    current_zone = None
    prob_map = {0.382:'Lower probability', 0.500:'Moderate probability',
                0.618:'HIGHEST probability (wholesale)'}
    for fib in sorted(FIB_LEVELS):
        if abs(current - levels[fib]) / levels[fib] < 0.003:
            current_zone = (fib, prob_map.get(fib,''))
            break

    # Invalid if closed below 61.8% with momentum
    fib_618 = levels[0.618]
    prev_close = bars['close'].iloc[-2]
    invalid = (prev_close > fib_618) and (current < fib_618 * 0.998)

    return {'levels': levels, 'current_zone': current_zone,
            'swing_low': swing_low, 'swing_high': swing_high,
            'setup_valid': not invalid, 'current_price': current}

fib = detect_fib_entry(bars)
print(f'Fibonacci Levels  (swing ${fib["swing_low"]:.2f} -> ${fib["swing_high"]:.2f})')
print(f'  0.0%  (top B)  = ${fib["levels"][0.0]}')
print(f'  38.2%          = ${fib["levels"][0.382]}')
print(f'  50.0%          = ${fib["levels"][0.500]}')
print(f'  61.8% (golden) = ${fib["levels"][0.618]}  <-- HIGHEST PROB')
print(f'  100%  (bot A)  = ${fib["levels"][1.0]}')
print(f'  Current price  = ${fib["current_price"]:.2f}')
if fib['current_zone']:
    print(f'  >>> Price in {fib["current_zone"][0]*100:.1f}% zone — {fib["current_zone"][1]}')
if not fib['setup_valid']:
    print('  INVALID: Closed below 61.8% with momentum. Setup cancelled.')

## 9. Trinity Play (A+ Setup — Highest Confidence)
**When Bounce 5 line + Fibonacci level + MA all converge at the SAME price zone.**
- All three must align within a tight price range (~0.2%)
- This is the A+ setup: increase position size
- Candlestick confirmation required at entry zone

In [ ]:
def detect_trinity(bars, b5_result, fib_result, season_result, tol=TRINITY_TOL):
    """
    Check if Bounce 5 resistance, a Fibonacci level, and an MA all converge.
    Returns dict: is_trinity, converging_levels, price_zone, confidence
    """
    res = {'is_trinity': False, 'converging_levels': [], 'price_zone': None}
    current = bars['close'].iloc[-1]
    latest  = bars.iloc[-1]

    # Collect candidate levels
    candidates = {}
    if b5_result.get('valid'):
        candidates['Bounce5_Resistance'] = b5_result['entry_price']
    for f, price in fib_result.get('levels', {}).items():
        if isinstance(f, float) and 0 < f < 1:
            candidates[f'Fib_{int(f*100)}pct'] = price
    candidates['MA50']  = latest['ma50']
    candidates['MA200'] = latest['ma200']

    # Find clusters within tolerance
    converging = []
    names      = list(candidates.keys())
    prices     = list(candidates.values())
    for i in range(len(names)):
        cluster = [names[i]]
        for j in range(len(names)):
            if i != j:
                if abs(prices[i]-prices[j])/prices[i] <= tol:
                    cluster.append(names[j])
        if len(cluster) >= 3:
            converging.append({'levels': cluster, 'price': prices[i]})

    if converging:
        best = max(converging, key=lambda x: len(x['levels']))
        near = abs(current - best['price']) / best['price'] < 0.005
        res.update({'is_trinity': True,
                    'converging_levels': best['levels'],
                    'price_zone': round(best['price'], 2),
                    'near_current_price': near,
                    'confidence': 'A+ — Increase position size!'})
    return res

trinity = detect_trinity(bars, b5, fib, season)
if trinity['is_trinity']:
    print('*** TRINITY PLAY DETECTED — A+ SETUP ***')
    print(f'  Converging at: ${trinity["price_zone"]}')
    print(f'  Levels: {", ".join(trinity["converging_levels"])}')
    print(f'  Confidence: {trinity["confidence"]}')
    if trinity.get('near_current_price'): print('  >>> Price is near the Trinity zone NOW!')
else:
    print('No Trinity convergence found. Wait for all three levels to align.')

## 10. Candlestick Confirmation
Use at entry zones to confirm direction before placing order.

**Bullish (confirm LONG):** Hammer, Bullish Engulfing, Morning Star, Piercing Pattern, Bullish Harami

**Bearish (confirm SHORT):** Hanging Man, Shooting Star, Bearish Engulfing, Evening Star, Dark Cloud Cover

**Doji at key level:** Potential reversal — wait for next candle confirmation

In [ ]:
def detect_candle_pattern(bars):
    """Detect last 2-bar candlestick pattern. Returns pattern name and direction."""
    if len(bars) < 2:
        return {'pattern':'Not enough data','direction':'none'}

    c   = bars.iloc[-1]    # current candle
    p   = bars.iloc[-2]    # previous candle
    body    = abs(c['close'] - c['open'])
    rng     = c['high'] - c['low']
    pbody   = abs(p['close'] - p['open'])

    # Doji
    if body / rng < 0.1 if rng > 0 else False:
        return {'pattern':'Doji','direction':'neutral','note':'Wait for next candle'}

    # Hammer (bullish)
    lower_wick = min(c['open'],c['close']) - c['low']
    upper_wick = c['high'] - max(c['open'],c['close'])
    if (lower_wick >= 2*body and upper_wick < body and c['close'] > c['open']):
        return {'pattern':'Hammer','direction':'bullish'}

    # Shooting Star (bearish)
    if (upper_wick >= 2*body and lower_wick < body and c['close'] < c['open']):
        return {'pattern':'Shooting Star','direction':'bearish'}

    # Bullish Engulfing
    if (p['close'] < p['open'] and c['close'] > c['open'] and
        c['open'] < p['close'] and c['close'] > p['open']):
        return {'pattern':'Bullish Engulfing','direction':'bullish'}

    # Bearish Engulfing
    if (p['close'] > p['open'] and c['close'] < c['open'] and
        c['open'] > p['close'] and c['close'] < p['open']):
        return {'pattern':'Bearish Engulfing','direction':'bearish'}

    # Hanging Man (bearish — appears in uptrend)
    if (lower_wick >= 2*body and upper_wick < body and c['close'] < c['open']):
        return {'pattern':'Hanging Man','direction':'bearish'}

    body_pct = body / c['close'] if c['close'] > 0 else 0
    if c['close'] > c['open']:
        return {'pattern':'Bullish candle','direction':'bullish'}
    else:
        return {'pattern':'Bearish candle','direction':'bearish'}

candle = detect_candle_pattern(bars)
print(f'Last candle pattern: {candle["pattern"]} ({candle["direction"]})')
if 'note' in candle: print(f'  Note: {candle["note"]}')

## 11. Trade Management & Order Execution

**Daily rules:**
- Daily goal: $100 (ES equivalent). Stop trading after hitting this.
- Max 2-3 trades per day
- Risk per trade: 0.4-0.5% of account
- Once up 1 point equivalent, move stop to breakeven (Panzy Stop)
- Trailing stop: manually move up as price moves in favor

In [ ]:
class TradeManager:
    """Tracks daily PnL, enforces daily limits, and manages positions."""

    def __init__(self):
        self.daily_pnl      = 0.0
        self.trades_today   = 0
        self.open_position  = None
        self.breakeven_set  = False

    def can_trade(self):
        if self.daily_pnl >= DAILY_GOAL_USD:
            print(f'DAILY GOAL HIT (${self.daily_pnl:.2f}). No more trades today.')
            return False
        if self.trades_today >= MAX_TRADES_PER_DAY:
            print(f'Max trades ({MAX_TRADES_PER_DAY}) reached.')
            return False
        if self.open_position is not None:
            print('Already in a position.')
            return False
        return True

    def calc_shares(self, account, entry, stop):
        """Position size based on risk % of account."""
        equity    = float(account.equity)
        risk_usd  = equity * RISK_PER_TRADE_PCT
        risk_pp   = abs(entry - stop)
        if risk_pp <= 0: return 1
        return max(1, int(risk_usd / risk_pp))

    def record_trade(self, pnl):
        self.daily_pnl    += pnl
        self.trades_today += 1
        self.open_position = None
        self.breakeven_set = False
        print(f'Trade closed. PnL: ${pnl:.2f} | Daily total: ${self.daily_pnl:.2f}')

mgr = TradeManager()
print(f'Trade manager ready. Can trade: {mgr.can_trade()}')

def place_short_order(symbol, qty, entry_price, stop_price, target_price, label='Bounce5'):
    """Place a short market order. Paper trading safe."""
    print(f'[{label}] Placing SHORT {qty} shares of {symbol}')
    print(f'  Entry: ~${entry_price:.2f} | Stop: ${stop_price:.2f} | Target: ${target_price:.2f}')
    order = trading_client.submit_order(
        MarketOrderRequest(
            symbol=symbol, qty=qty,
            side=OrderSide.SELL,
            time_in_force=TimeInForce.DAY
        )
    )
    print(f'  Order submitted: {order.id}')
    return order

def place_long_order(symbol, qty, entry_price, stop_price, target_price, label='Predator'):
    """Place a long market order. Paper trading safe."""
    print(f'[{label}] Placing LONG {qty} shares of {symbol}')
    print(f'  Entry: ~${entry_price:.2f} | Stop: ${stop_price:.2f} | Target: ${target_price:.2f}')
    order = trading_client.submit_order(
        MarketOrderRequest(
            symbol=symbol, qty=qty,
            side=OrderSide.BUY,
            time_in_force=TimeInForce.DAY
        )
    )
    print(f'  Order submitted: {order.id}')
    return order

print('Order functions ready. Uncomment execute lines to go live.')

## 12. Full Scanner — Run All Strategies
This cell runs everything and prints a consolidated signal report.

In [ ]:
def run_scanner(symbol=SYMBOL, timeframe=TIMEFRAME):
    """Full signal scanner. Run this each bar to get trade signals."""
    print('=' * 60)
    print(f' SCANNER | {symbol} | {timeframe} | {datetime.datetime.now().strftime("%H:%M:%S")}')
    print('=' * 60)

    # 1. Get data
    bars = get_bars(symbol, timeframe)
    latest = bars.iloc[-1]

    # 2. Season
    s = detect_season(bars)
    print(f'\n[SEASON] {s["name"]} | {s["description"]}')
    if s['golden_cross']: print('  *** GOLDEN CROSS!')
    if s['death_cross']:  print('  *** DEATH CROSS!')

    # 3. MA Rules
    above50  = latest['close'] > latest['ma50']
    above200 = latest['close'] > latest['ma200']
    print(f'\n[MAs] Price={latest["close"]:.2f} | MA50={latest["ma50"]:.2f} | MA200={latest["ma200"]:.2f}')
    print(f'      50MA: {"ABOVE (prefer longs)" if above50 else "BELOW (prefer shorts)"}')
    print(f'     200MA: {"ABOVE (bullish bias)" if above200 else "BELOW (bearish bias)"}')

    # 4. Bounce 5
    b5 = detect_bounce5(bars)
    print(f'\n[BOUNCE 5]', end=' ')
    if s['season'] != 1:
        print(f'SKIP — Season {s["season"]} not ideal for Bounce 5')
    elif b5['valid']:
        print(f'SIGNAL! SHORT@{b5["entry_price"]} | Target:{b5["target_price"]} | Stop:{b5["stop_price"]} | R:R {b5["risk_reward"]}:1')
        for w in b5['warnings']: print(f'  {w}')
    else:
        print(f'No setup — {b5["reason"]}')

    # 5. Predator
    pred = detect_predator_play(bars)
    print(f'\n[PREDATOR]', end=' ')
    if pred['valid']:
        print(f'SIGNAL! LONG@{pred["entry_price"]} | Target:{pred["target_price"]} | Stop:{pred["stop_price"]} | R:R {pred["risk_reward"]}:1')
        if pred['near_entry']: print('  >>> NEAR ENTRY NOW')
    else:
        print(f'No setup — {pred["reason"]}')

    # 6. Fibonacci
    fib = detect_fib_entry(bars)
    print(f'\n[FIBONACCI] 38.2%=${fib["levels"][0.382]} | 50%=${fib["levels"][0.500]} | 61.8%={fib["levels"][0.618]}')
    if fib['current_zone']:
        print(f'  >>> Price at {fib["current_zone"][0]*100:.1f}% — {fib["current_zone"][1]}')
    if not fib['setup_valid']:
        print('  INVALID: closed below 61.8%')

    # 7. Trinity
    trinity = detect_trinity(bars, b5, fib, s)
    print(f'\n[TRINITY]', end=' ')
    if trinity['is_trinity']:
        print(f'*** A+ SETUP at ${trinity["price_zone"]} | Levels: {", ".join(trinity["converging_levels"])}')
    else:
        print('No convergence')

    # 8. Candle confirmation
    candle = detect_candle_pattern(bars)
    print(f'\n[CANDLE] {candle["pattern"]} ({candle["direction"]})')

    # 9. Season 3 warning
    if s['season'] == 3:
        print('\n*** WHIPSAW — STAND ASIDE or reduce size significantly ***')

    print('\n' + '=' * 60)
    return {'season':s, 'bounce5':b5, 'predator':pred, 'fibonacci':fib, 'trinity':trinity, 'candle':candle}

# Run the scanner
signals = run_scanner()

## 13. Live Trading Loop (Paper Trading)
> Run this cell to scan continuously every minute. Stop the cell to pause.

In [ ]:
import time as _time

SCAN_INTERVAL_SEC = 60  # scan every 60 seconds (1 min bar)
LIVE_MODE = False        # set to True to actually place orders

mgr = TradeManager()

print('Starting live scanner. Press Stop to pause.')
print(f'Live mode: {LIVE_MODE}')

try:
    while True:
        now = datetime.datetime.now()
        # Only trade during market hours (9:30 AM - 4:00 PM ET)
        market_open  = now.replace(hour=9,  minute=30, second=0)
        market_close = now.replace(hour=16, minute=0,  second=0)

        if not (market_open <= now <= market_close):
            print(f'Market closed ({now.strftime("%H:%M")}). Waiting...')
            _time.sleep(60)
            continue

        if not mgr.can_trade():
            _time.sleep(SCAN_INTERVAL_SEC)
            continue

        signals = run_scanner()
        s       = signals['season']
        b5      = signals['bounce5']
        pred    = signals['predator']
        candle  = signals['candle']
        trinity = signals['trinity']

        # Priority 1: Trinity A+ setup
        if trinity['is_trinity'] and trinity.get('near_current_price'):
            print('>>> TRINITY A+ SETUP — consider increased size')

        # Priority 2: Bounce 5 (Season 1 only)
        if s['season'] == 1 and b5['valid'] and candle['direction'] == 'bearish':
            account = trading_client.get_account()
            qty = mgr.calc_shares(account, b5['entry_price'], b5['stop_price'])
            print(f'>>> BOUNCE 5 SIGNAL: SHORT {qty} {SYMBOL} | Candle confirmed: {candle["pattern"]}')
            if LIVE_MODE:
                order = place_short_order(SYMBOL, qty,
                            b5['entry_price'], b5['stop_price'], b5['target_price'])
                mgr.open_position = {'order': order, 'entry': b5['entry_price'],
                                     'stop': b5['stop_price'], 'target': b5['target_price'],
                                     'side': 'short'}

        # Priority 3: Predator Play
        elif pred['valid'] and pred.get('near_entry') and candle['direction'] == 'bullish':
            account = trading_client.get_account()
            qty = mgr.calc_shares(account, pred['entry_price'], pred['stop_price'])
            print(f'>>> PREDATOR SIGNAL: LONG {qty} {SYMBOL} | Candle confirmed: {candle["pattern"]}')
            if LIVE_MODE:
                order = place_long_order(SYMBOL, qty,
                            pred['entry_price'], pred['stop_price'], pred['target_price'])
                mgr.open_position = {'order': order, 'entry': pred['entry_price'],
                                     'stop': pred['stop_price'], 'target': pred['target_price'],
                                     'side': 'long'}

        # Season 3: stand aside
        elif s['season'] == 3:
            print('WHIPSAW season — standing aside')

        _time.sleep(SCAN_INTERVAL_SEC)

except KeyboardInterrupt:
    print('Scanner stopped.')

## Notes & Next Steps

**Open items to refine (from Dave's content):**
- Exact stop loss point distances per strategy
- Full Trinity Play confluence rules
- Crack Run setup (referenced but not yet documented)

**Before going live:**
1. Run on paper trading for at least 2-4 weeks
2. Track every trade in a journal (entry, exit, R:R, season, pattern)
3. Verify the season detector matches what you see on your chart manually
4. Tune `slope_thresh` and `sep_thresh` in `detect_season()` to match your timeframe
5. Set `LIVE_MODE = True` only after you're confident in the signals